# Compare Generative Models: Synthesis and Model Selection

This final activity notebook compares the five model families developed in notebooks 01-12. It requires **no retraining, checkpoints, data download, or expensive execution**. Work from memory first, then use the links and supplied reference discussion to revise your answers.

### Learning goals

By the end, you should be able to:

1. distinguish the objectives and computational trade-offs of VAEs, flows, autoregressive models, GANs, and diffusion models;
2. separate exact likelihood, reconstruction, inversion, and sample quality rather than treating them as synonyms;
3. select a model family from application constraints and defend the choice;
4. identify what a proposed evaluation metric does and does not establish; and
5. state one important failure mode for each family.

In [1]:
import matplotlib.pyplot as plt

PLOT_THEME = "light"  # Change to "dark" for the dark plot palette.
if PLOT_THEME not in {"light", "dark"}:
    raise ValueError("PLOT_THEME must be 'light' or 'dark'.")

# Start from Matplotlib defaults so light mode keeps the standard coloring.
plt.style.use("default")
if PLOT_THEME == "dark":
    plt.style.use("dark_background")
    plt.rcParams.update(
        {
            "figure.facecolor": "#0a0f18",
            "figure.edgecolor": "#0a0f18",
            "axes.facecolor": "#111926",
            "axes.edgecolor": "#e8eff9",
            "axes.labelcolor": "#e8eff9",
            "axes.titlecolor": "#e8eff9",
            "text.color": "#e8eff9",
            "xtick.color": "#e8eff9",
            "ytick.color": "#e8eff9",
            "grid.color": "#2b3340",
            "savefig.facecolor": "#0a0f18",
            "savefig.edgecolor": "#0a0f18",
        }
    )

PLOT_REFERENCE_COLOR = "black" if PLOT_THEME == "light" else "#6cd2e8"
PLOT_GENERATED_COLOR = "tab:orange" if PLOT_THEME == "light" else "#f09a4b"
PLOT_GUIDE_COLOR = "black" if PLOT_THEME == "light" else "#e8eff9"

plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})


## 1. Retrieval practice: answer before reopening earlier notebooks

Write one or two sentences for each prompt. Mark uncertain answers with a question mark rather than looking them up immediately.

1. Why is a VAE objective a lower bound rather than an exact marginal log likelihood?
2. What two terms appear in the flow change-of-variables formula?
3. Why can an autoregressive model train on all observed positions together but not generate all positions together?
4. What evidence would reveal GAN mode dropping on the ring of Gaussians?
5. Why can a diffusion model train with one randomly selected noise level per example even though sampling uses many steps?
6. In notebook 11, why is it important to hold the denoiser fixed when comparing samplers?
7. In classifier-free guidance, what usually happens to diversity as guidance strength becomes very large?

**Your retrieval notes**

- 1: _..._
- 2: _..._
- 3: _..._
- 4: _..._
- 5: _..._
- 6: _..._
- 7: _..._

## 2. Learner comparison activity

Complete the blank or partial cells before reading the reference comparison. Use short phrases and add qualifications where the answer depends on architecture.

| Family | Training signal | Exact normalized likelihood? | Training vs. sampling parallelism | Reconstruction / inversion | One strength | One failure mode or cost |
|---|---|---|---|---|---|---|
| VAE | ELBO = reconstruction term + ___ | ___ | Training: parallel. Sampling: ___ | Approximate reconstruction through ___ | Structured latent space | ___ |
| Normalizing flow | ___ via change of variables | Yes, under modeling assumptions | Architecture-dependent, often parallel in coupling flows | ___ | ___ | Invertibility/Jacobian constraints |
| Autoregressive | Sum of conditional ___ | ___ | Training: teacher-forced. Sampling: ___ | Prefix completion, but no compact latent inverse | ___ | ___ |
| GAN | ___ game between generator and discriminator/critic | No | Minibatches parallel. Usually one-pass generation | ___ | Sharp, fast samples | ___ |
| Diffusion | Predict noise/score across ___ | ___ | Training examples parallel. Sampling steps ___ | Forward corruption known. Inversion/reconstruction ___ | Stable objective and strong coverage | ___ |

### Two cross-family claims to test

For each claim, write **agree**, **disagree**, or **it depends**, followed by a reason.

1. “A model with the best test likelihood must produce the most perceptually convincing samples.”
2. “A deterministic map from latent noise always provides a meaningful inverse for a real observation.”

## 3. Reference comparison and discussion

Use this section to correct or qualify your table, not merely to copy it.

| Family | Objective and likelihood | Compute pattern | Reconstruction / inversion | Characteristic trade-off |
|---|---|---|---|---|
| VAE | Maximizes an ELBO containing an expected reconstruction log likelihood and a posterior-to-prior KL. The marginal likelihood is generally not exact. | Training and one-pass decoding are parallel across a batch. | The encoder gives a fast approximate posterior and reconstruction path, not an exact inverse. | Stable representation learning and fast sampling versus posterior collapse, prior mismatch, and smoothing from simple likelihoods. |
| Normalizing flow | Maximizes exact likelihood through a base-density term plus a log absolute Jacobian determinant. | Coupling-based designs permit substantial parallelism in both directions. | A bijection provides exact encode/decode inversion up to numerical precision. | Exact density and inversion versus restrictive architecture, equal dimensions, and Jacobian/memory cost. |
| Autoregressive | Maximizes exact likelihood by summing log conditional probabilities in a chosen ordering. | Observed-token conditionals can be teacher-forced together. Generation must reveal one token or coordinate after another. | It scores complete data and completes prefixes, but has no native compact latent inverse. | Flexible density estimation versus sequential sampling and ordering/attention costs. |
| GAN | Optimizes an adversarial game. The generator defines an implicit distribution without tractable likelihood. | Alternating training is delicate, while generation is usually one parallel network pass. | A basic GAN has no encoder. Inversion needs optimization or an added learned encoder. | Sharp, fast samples versus instability, mode dropping, and weak likelihood-based diagnostics. |
| Diffusion | Learns a denoising score/noise target related to variational objectives. The standard simple loss is not a directly evaluated exact likelihood. | Noise levels are sampled independently during training. Teverse-time sampling remains sequential across solver steps. | Forward noising has a closed form, but reconstructing a particular observation is not a native exact inverse. | Stable training, coverage, and conditioning versus iterative sampling cost and solver/guidance sensitivity. |

The two claims above are both unsafe. Likelihood rewards probability assignment under the observation model and can disagree with perceptual judgments. A generator maps latent points to samples, but that does not imply that every real observation has a unique, recoverable latent code.

## 4. Scenario-based model selection

For each scenario, choose a primary family, name a plausible alternative, and state the deciding constraint. More than one answer can be defensible. The reasoning matters.

### A. Real-time generation
A client needs thousands of independent samples with very low latency. Exact likelihood is not required, and offline training can be expensive. Which family would you start with? What risk must the evaluation specifically test?

_Your answer: ..._

### B. Exact density and reversible representation
A scientific workflow requires a normalized likelihood and a deterministic round trip between observations and same-dimensional latent variables. Which family directly exposes both? What modeling constraint follows?

_Your answer: ..._

### C. Missing suffix completion
A sequence task requires calibrated next-token probabilities and completion from arbitrary observed prefixes. Generation latency is acceptable. Which family is the natural baseline?

_Your answer: ..._

### D. Editable latent representation
A classroom project values fast encoding, reconstruction, interpolation, and a compact latent representation more than exact likelihood or maximum sharpness. Which family is the simplest fit, and what latent-space pathology should be checked?

_Your answer: ..._

### E. Conditional image generation
A team wants high-quality class-conditional images and can spend multiple network evaluations per sample. It also wants a user-facing fidelity/diversity control. Which mechanism from this course provides that control, and why should the scale be swept rather than maximized?

_Your answer: ..._


## 5. Evaluation metrics: useful evidence, not a single scoreboard

A responsible comparison starts by naming the property of interest.

- **Held-out NLL / bits per dimension** tests probability assignment only for models with comparable, well-defined observation models. Dequantization and discretization choices matter, and good likelihood does not automatically imply good perceptual samples.
- **ELBO or denoising loss** is a training diagnostic, not an automatically comparable cross-family score. Its scale and tightness depend on the formulation.
- **FID-like feature distances** mix fidelity and coverage in a chosen feature space. They are biased at small sample counts, depend on the feature extractor, and can miss task-specific errors. MNIST classifier features are not an ImageNet-quality universal perceptual space.
- **Precision/recall or density/coverage metrics** separate some quality and diversity effects, but still inherit feature-space, neighborhood, and sample-size choices.
- **Classifier accuracy or confidence on generated digits** measures recognizability to that classifier. It can reward prototypical or adversarial artifacts and says little about within-class diversity.
- **Mode counts and mode-frequency total variation** are informative on the known eight-mode ring. They do not generalize automatically to data without known discrete modes.
- **Reconstruction error** is meaningful for models with an inference or inversion path. Comparing a VAE reconstruction directly with an unconditional GAN sample answers different questions.
- **Wall-clock time** depends on hardware, batch size, software, and synchronization. Report those conditions and pair time with hardware-independent counts such as parameter count or network-function evaluations.
- **Human inspection** can reveal artifacts and semantics but is subjective. Use blinded protocols and multiple raters for serious claims.

### Metric critique

A report claims: “Model X wins because its 100 generated MNIST digits have the highest classifier confidence.” List at least three missing checks or caveats.

1. _..._
2. _..._
3. _..._

## 6. Return to the evidence

Use these links only after attempting retrieval.

### Shared ring-of-Gaussians laboratory

- [01 — VAE: ELBO, reparameterization, reconstruction, and latent space](01_vae_ring_of_gaussians.ipynb)
- [02 — Flow: change of variables, Jacobians, exact likelihood, and inversion](02_normalizing_flow_ring_of_gaussians.ipynb)
- [03 — Autoregressive: normalized conditionals, exact likelihood, and sequential generation](03_autoregressive_ring_of_gaussians.ipynb)
- [04 — GAN: adversarial objectives, discriminator geometry, and mode collapse](04_gan_ring_of_gaussians.ipynb)
- [05 — Diffusion: forward corruption, noise prediction, and reverse trajectories](05_diffusion_ring_of_gaussians.ipynb)

### MNIST scale-up and extensions

- [06 — Convolutional VAE](06_vae_mnist.ipynb)
- [07 — Multiscale Glow](07_normalizing_flow_mnist.ipynb)
- [08 — Autoregressive image Transformer](08_autoregressive_mnist.ipynb)
- [09 — Spectral-normalized ResNet GAN](09_gan_mnist.ipynb)
- [10 — DDPM with a U-Net](10_diffusion_mnist.ipynb)
- [11 — DDPM, DDIM, and DPM-Solver++ sampling](11_diffusion_sampling_mnist.ipynb)
- [12 — Classifier-free guidance](12_diffusion_conditional_mnist.ipynb)

## 7. Self-check and exit ticket

Without using the reference table, answer each item in one sentence.

1. Name the two families here that directly provide exact normalized likelihoods in their standard course formulations.
2. Which family has the clearest native exact inverse, and what structural requirement enables it?
3. Contrast autoregressive and diffusion sampling in terms of what must happen sequentially.
4. Give one reason a GAN can produce convincing samples while missing part of the data distribution.
5. Give one reason a high guidance scale can make a conditional diffusion result worse.
6. Name two metrics you would report together for a model-selection scenario above, and explain why they complement one another.

### Confidence check

Rate each outcome from 1 (not yet) to 4 (can teach it):

- I can connect each family to its objective: **__/4**
- I can compare training and sampling parallelism: **__/4**
- I can distinguish likelihood, inversion, and reconstruction: **__/4**
- I can defend a model choice using constraints: **__/4**
- I can challenge an overbroad metric claim: **__/4**

>**Exit ticket:** Name one belief you changed during this course, one trade-off you can now explain, and one question you would investigate next.

## 8. References

- D. P. Kingma and M. Welling, *Auto-Encoding Variational Bayes*, 2014.
- D. Rezende and S. Mohamed, *Variational Inference with Normalizing Flows*, 2015.
- L. Dinh, J. Sohl-Dickstein, and S. Bengio, *Density Estimation Using Real NVP*, 2017.
- D. P. Kingma and P. Dhariwal, *Glow: Generative Flow with Invertible 1x1 Convolutions*, 2018.
- A. van den Oord et al., *Conditional Image Generation with PixelCNN Decoders*, 2016.
- I. Goodfellow et al., *Generative Adversarial Nets*, 2014.
- J. Ho, A. Jain, and P. Abbeel, *Denoising Diffusion Probabilistic Models*, 2020.
- J. Song, C. Meng, and S. Ermon, *Denoising Diffusion Implicit Models*, 2021.
- J. Ho and T. Salimans, *Classifier-Free Diffusion Guidance*, 2022.
- T. Karras et al., *Elucidating the Design Space of Diffusion-Based Generative Models*, 2022.

## Final takeaway

There is no context-free best generative model. Choose by matching the objective, representation, sampling process, compute budget, and failure modes to the application. Always support the choice with multiple metrics that measure the properties you actually need.